## Load weather data from open-meteo with API request

* fetch weather data of the selected locations from 2019-01-01 till yesterday over open-meteo archive API
* get the latitude and longitude of the selected locations
* construct the request with header information with paramaters including the following
	- latitude and longitude of the selected locations
	- start and end date time, hourly, the weather variables to be requested
	- PV
		- shortwave_radiation, direct_radiation, diffuse_radiation, cloud_cover
	- Wind 
		- wind_speed_100m, wind_direction_100m
* calculate weighted weather data and ingest into database

In [7]:
# Set a consistent style for all plots
import matplotlib.pyplot as plt
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})

## Fetch weather data from open-meteo


In [1]:
from pathlib import Path
import os
import sys
import pandas as pd

# Ensure project modules are importable when notebook runs from notebook/
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('../util'))
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    SELECTED_CITIES,
    CITY_POPULATION,
    PV_WEATHER_VARIABLES,
    WIND_WEATHER_VARIABLES,
    PV_CLUSTER_LOCATIONS,
    WIND_CLUSTER_LOCATIONS,
    PV_WEATHER_SERIES_IDS,
    WIND_WEATHER_SERIES_IDS,
 )
from util.weather_weighted import (
    build_yearly_weights,
    fetch_weighted_weather_for_technology,
 )

# Keep this range short for fast iteration during development.
START_DATE = "2025-01-01"
END_DATE = "2025-01-03"

PV_CAPACITY_CSV = Path("../data/processed/pv_cluster_yearly_capacity_since_2019.csv")
WIND_CAPACITY_CSV = Path("../data/processed/wind_cluster_yearly_capacity_since_2019.csv")

PV_OUTPUT_CSV = Path("../data/processed/pv_weather_weighted_hourly.csv")
WIND_OUTPUT_CSV = Path("../data/processed/wind_weather_weighted_hourly.csv")

In [2]:
pv_weights_by_year = build_yearly_weights(PV_CAPACITY_CSV, technology_prefix="pv")
wind_weights_by_year = build_yearly_weights(WIND_CAPACITY_CSV, technology_prefix="wind")

df_weather_pv = fetch_weighted_weather_for_technology(
    technology_name="pv",
    locations=PV_CLUSTER_LOCATIONS,
    weather_variables=PV_WEATHER_VARIABLES,
    weights_by_year=pv_weights_by_year,
    start_date=START_DATE,
    end_date=END_DATE,
    selected_cities=SELECTED_CITIES,
    city_population=CITY_POPULATION,
    city_sleep=0.1,
 )

df_weather_wind = fetch_weighted_weather_for_technology(
    technology_name="wind",
    locations=WIND_CLUSTER_LOCATIONS,
    weather_variables=WIND_WEATHER_VARIABLES,
    weights_by_year=wind_weights_by_year,
    start_date=START_DATE,
    end_date=END_DATE,
    selected_cities=SELECTED_CITIES,
    city_population=CITY_POPULATION,
    city_sleep=0.1,
 )

PV_OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_weather_pv.to_csv(PV_OUTPUT_CSV, index=False)
df_weather_wind.to_csv(WIND_OUTPUT_CSV, index=False)

print("PV rows:", len(df_weather_pv), "| Wind rows:", len(df_weather_wind))
print("PV columns:", list(df_weather_pv.columns))
print("Wind columns:", list(df_weather_wind.columns))
display(df_weather_pv.head(3))
display(df_weather_wind.head(3))

PV rows: 72 | Wind rows: 72
PV columns: ['time', 'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'cloud_cover']
Wind columns: ['time', 'wind_speed_100m', 'wind_direction_100m']


,time,shortwave_radiation,direct_radiation,diffuse_radiation,cloud_cover
0,2025-01-01 00:00:00+01:00,0.0,0.0,0.0,61.385740
1,2025-01-01 01:00:00+01:00,0.0,0.0,0.0,70.459967
2,2025-01-01 02:00:00+01:00,0.0,0.0,0.0,72.353082


,time,wind_speed_100m,wind_direction_100m
0,2025-01-01 00:00:00+01:00,55.376052,226.745192
1,2025-01-01 01:00:00+01:00,57.176116,228.284297
2,2025-01-01 02:00:00+01:00,59.263985,226.137033


In [6]:
# Update database with new weather data and inspect stored weather series
from src.etl_price import update_database
import sqlalchemy as sa

update_database()

pv_ids = tuple(PV_WEATHER_SERIES_IDS.values())
wind_ids = tuple(WIND_WEATHER_SERIES_IDS.values())

query_pv = f"""
    SELECT time, series_id, value
    FROM timeseries_values
    WHERE series_id IN {pv_ids}
    ORDER BY time DESC, series_id
    LIMIT 168;
    """
query_wind = f"""
    SELECT time, series_id, value
    FROM timeseries_values
    WHERE series_id IN {wind_ids}
    ORDER BY time DESC, series_id
    LIMIT 168;
    """

engine = sa.create_engine("sqlite:///../db/energy_demand.db")
with engine.connect() as conn:
    df_agg_pv = pd.read_sql(query_pv, conn)
    df_agg_wind = pd.read_sql(query_wind, conn)

display(df_agg_pv.head())
display(df_agg_wind.head())

Series catalog seeded/updated rows: 11

Current SMARD data status:
  price_de_lu_eur_mwh         :  65015 rows | max: 2026-06-01T21:00:00Z
  gen_wind_onshore_mwh        :  65007 rows | max: 2026-06-01T13:00:00Z
  gen_wind_offshore_mwh       :  65007 rows | max: 2026-06-01T13:00:00Z
  gen_pv_mwh                  :  65007 rows | max: 2026-06-01T13:00:00Z
  gen_other_conventional_mwh  :  65007 rows | max: 2026-06-01T13:00:00Z

SMARD series up to date — skip SMARD fetch.

Current Open-Meteo data status:
  pv_weather_shortwave_radiation:  64991 rows | max: 2026-05-31T21:00:00Z
  pv_weather_direct_radiation :  64991 rows | max: 2026-05-31T21:00:00Z
  pv_weather_diffuse_radiation:  64991 rows | max: 2026-05-31T21:00:00Z
  pv_weather_cloud_cover      :  64991 rows | max: 2026-05-31T21:00:00Z
  wind_weather_wind_speed_100m:  64991 rows | max: 2026-05-31T21:00:00Z
  wind_weather_wind_direction_100m:  64991 rows | max: 2026-05-31T21:00:00Z

Open-Meteo weather is up to date (target: yesterday).

D

,time,series_id,value
0,2026-05-31T21:00:00Z,pv_weather_cloud_cover,78.009474
1,2026-05-31T21:00:00Z,pv_weather_diffuse_radiation,0.000000
2,2026-05-31T21:00:00Z,pv_weather_direct_radiation,0.000000
3,2026-05-31T21:00:00Z,pv_weather_shortwave_radiation,0.000000
4,2026-05-31T20:00:00Z,pv_weather_cloud_cover,81.856874


,time,series_id,value
0,2026-05-31T21:00:00Z,wind_weather_wind_direction_100m,193.891646
1,2026-05-31T21:00:00Z,wind_weather_wind_speed_100m,14.028812
2,2026-05-31T20:00:00Z,wind_weather_wind_direction_100m,200.778387
3,2026-05-31T20:00:00Z,wind_weather_wind_speed_100m,12.727964
4,2026-05-31T19:00:00Z,wind_weather_wind_direction_100m,229.839676
